# Модуль 7, ДЗ 1 — RAG, который работает

> Если закрыли лекцию: это [Модуль 7. RAG, который работает](https://itrubnikov.github.io/Train_of_Thought/docs/modules/07-rag/) курса «От нуля до своих агентов».

Здесь вы соберёте полный RAG-пайплайн по маленькой базе знаний и пройдёте его слой за слоем — ровно в том порядке, в котором лекция вводит понятия:

- **chunk → embedding → cosine-поиск** — наивный baseline на numpy (та самая «геометрия смыслов»);
- **Chroma** — то же хранилище, но по-инженерному: персистентность, метаданные, ANN-индекс;
- **hybrid (BM25 + dense)** — почему чисто векторный поиск мажет на точных токенах и как это чинит BM25;
- **reranking (cross-encoder)** — переупорядочиваем top-N кандидатов в точный top-k;
- **generation** — собираем контекст в промпт, Claude отвечает только по нему и ссылается на источник, а на вопрос вне базы честно говорит «не знаю».

**Главное про запуск.** Весь retrieval (chunk → embed → store → hybrid → rerank) работает **локально и без ключа** — на `sentence-transformers`, `chromadb`, `rank-bm25`. Ключ нужен только последнему шагу (генерация ответа через Claude). Если ключа нет — этот шаг аккуратно пропускается с понятным сообщением, и `Run all` всё равно проходит до конца.

CPU достаточно, GPU не нужен. Первый запуск качает модели эмбеддингов и reranker (несколько сотен мегабайт) — это нормально.

## 0. Установка и импорты

Ставим локальные библиотеки для retrieval плюс `anthropic` для финального шага генерации. Все модели — CPU-friendly, ключ на этом шаге не нужен.

In [ ]:
!pip install -q sentence-transformers chromadb rank-bm25 anthropic numpy && echo "[ok] зависимости установлены"

In [ ]:
import os
import numpy as np

from sentence_transformers import SentenceTransformer, CrossEncoder

# Embedding-модель: текст -> вектор из 384 чисел. Лёгкая, многоязычная, без ключа.
# Первый запуск качает веса (~80 МБ), дальше берёт из кэша.
EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

print("embedding-модель загружена:", EMB_MODEL_NAME)
print("размерность вектора:", embedder.get_sentence_embedding_dimension())

## 1. База знаний

Маленькая FAQ службы поддержки интернет-магазина: оплата, доставка, возврат, аккаунт. В жизни вы режете большие документы на чанки (куски по 200-500 токенов с перекрытием); тут уже готовые короткие куски, чтобы сосредоточиться на самом retrieval.

Каждый чанк — это `id + текст + метаданные`. Метаданные (`source`, `topic`) нужны, чтобы в ответе дать ссылку на источник и при желании отфильтровать поиск. Обратите внимание на чанк про **ошибку E-451** и про артикул **SKU-90210** — на них чуть позже сломается чисто векторный поиск.

In [ ]:
# id + текст чанка + метаданные. Это и есть "идеальная запись" из лекции
# (вектор посчитаем отдельно — embedding-моделью).
KB = [
    {"id": "pay-01", "text": "Оплатить заказ можно банковской картой или через СБП по QR-коду.", "source": "faq/payments.md", "topic": "оплата"},
    {"id": "pay-02", "text": "Оплата частями (рассрочка) доступна для заказов от 5000 рублей через сервис Долями.", "source": "faq/payments.md", "topic": "оплата"},
    {"id": "pay-03", "text": "Если при оплате картой возникает ошибка E-451, банк отклонил транзакцию из-за лимита по карте. Обратитесь в свой банк или оплатите через СБП.", "source": "faq/payments.md", "topic": "оплата"},
    {"id": "pay-04", "text": "Чек об оплате приходит на email в течение 15 минут после успешной транзакции.", "source": "faq/payments.md", "topic": "оплата"},
    {"id": "del-01", "text": "Бесплатная доставка курьером действует на заказы от 3000 рублей, иначе доставка стоит 350 рублей.", "source": "faq/delivery.md", "topic": "доставка"},
    {"id": "del-02", "text": "Курьер созванивается за час до доставки и привозит заказ в выбранный двухчасовой интервал.", "source": "faq/delivery.md", "topic": "доставка"},
    {"id": "del-03", "text": "Самовывоз из пунктов выдачи доступен на следующий день после оформления заказа.", "source": "faq/delivery.md", "topic": "доставка"},
    {"id": "del-04", "text": "Отследить посылку можно по трек-номеру в личном кабинете в разделе Мои заказы.", "source": "faq/delivery.md", "topic": "доставка"},
    {"id": "ret-01", "text": "Вернуть товар надлежащего качества можно в течение 14 дней с момента получения.", "source": "faq/returns.md", "topic": "возврат"},
    {"id": "ret-02", "text": "Деньги за возврат приходят на ту же карту, которой оплачивали, в течение 3-10 рабочих дней.", "source": "faq/returns.md", "topic": "возврат"},
    {"id": "ret-03", "text": "Чтобы оформить возврат, заполните заявку в личном кабинете и приложите фото товара.", "source": "faq/returns.md", "topic": "возврат"},
    {"id": "ret-04", "text": "Товары из категории нижнее бельё и парфюмерия возврату не подлежат по закону.", "source": "faq/returns.md", "topic": "возврат"},
    {"id": "acc-01", "text": "Если SMS-код для входа не приходит, запросите вход по ссылке на email.", "source": "faq/account.md", "topic": "аккаунт"},
    {"id": "acc-02", "text": "Сменить номер телефона в аккаунте можно в настройках профиля после подтверждения старого номера.", "source": "faq/account.md", "topic": "аккаунт"},
    {"id": "acc-03", "text": "Удалить аккаунт можно через поддержку; данные удаляются безвозвратно в течение 30 дней.", "source": "faq/account.md", "topic": "аккаунт"},
    {"id": "acc-04", "text": "Бонусные баллы начисляются в размере 5 процентов от суммы заказа и действуют 12 месяцев.", "source": "faq/account.md", "topic": "аккаунт"},
    {"id": "prod-01", "text": "Беспроводные наушники TrueSound Air, артикул SKU-90210, поддерживают активное шумоподавление и работают до 30 часов.", "source": "catalog/audio.md", "topic": "товары"},
    {"id": "prod-02", "text": "Гарантия на электронику составляет 12 месяцев с момента покупки и распространяется на заводской брак.", "source": "catalog/audio.md", "topic": "товары"},
    {"id": "prod-03", "text": "Умная колонка HomeVoice Mini управляет умным домом голосом и стоит 4990 рублей.", "source": "catalog/audio.md", "topic": "товары"},
    {"id": "prod-04", "text": "Промокод WELCOME10 даёт скидку 10 процентов на первый заказ и не суммируется с другими акциями.", "source": "catalog/audio.md", "topic": "товары"},
]

DOCS = [c["text"] for c in KB]
IDS = [c["id"] for c in KB]
print(f"в базе {len(KB)} чанков, тем: {sorted({c['topic'] for c in KB})}")

## 2. Эмбеддинги + наивный cosine-поиск (numpy)

Baseline из лекции: считаем по вектору на каждый чанк, нормируем — тогда скалярное произведение и есть **cosine similarity**. Поиск — это `top-k` ближайших векторов к вектору запроса. Никакой БД, чистая геометрия в numpy.

In [ ]:
# Вектор на каждый чанк. normalize_embeddings=True -> длина вектора 1,
# тогда (emb @ q) == cosine similarity.
EMB = embedder.encode(DOCS, normalize_embeddings=True)
print("матрица эмбеддингов:", EMB.shape, "(чанков, размерность)")


def dense_search(query, k=3):
    """top-k чанков по cosine similarity. Возвращает list[(id, text, score)]."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = EMB @ q                       # cosine, т.к. всё нормировано
    top = np.argsort(-scores)[:k]
    return [(IDS[i], DOCS[i], float(scores[i])) for i in top]


def show(results):
    for cid, text, score in results:
        print(f"  {score:+.3f}  [{cid}]  {text}")


for q in ["как вернуть деньги за товар", "сколько стоит доставка"]:
    print(f"\nЗАПРОС: {q!r}")
    show(dense_search(q, k=3))

Наверх всплывают чанки про возврат/деньги и про стоимость доставки, хотя слова в запросе и в чанках разные — это и есть поиск по смыслу. Минусы у такого хранилища ровно те, что в лекции: всё живёт в оперативке (перезапуск — и данных нет), поиск линейным перебором, нет метаданных и фильтров. Дальше перепишем это на настоящую БД.

## 3. Хранилище по-инженерному: Chroma

Тот же поиск, но через `chromadb.PersistentClient`: данные ложатся на диск (переживут перезапуск), рядом с вектором хранятся текст и метаданные, под капотом ANN-индекс. Мы передаём Chroma уже посчитанные эмбеддинги (`embeddings=...`) той же моделью — чтобы запрос и база жили в одном пространстве.

Сравните с numpy-версией: исчезли ручной `argsort` и хранение векторов руками, появились персистентность, метаданные и фильтры. Это и есть «что даёт настоящая БД» — не лучшую релевантность, а инфраструктуру вокруг поиска.

In [ ]:
import chromadb

# PersistentClient пишет на диск (в Colab/Kaggle это /kaggle/working или текущая папка).
client_db = chromadb.PersistentClient(path="./kb_chroma")

# get_or_create + чистка, чтобы повторный Run all не падал на дублях id.
col = client_db.get_or_create_collection("support", metadata={"hnsw:space": "cosine"})
if col.count() > 0:
    col.delete(ids=IDS)

col.add(
    ids=IDS,
    documents=DOCS,
    metadatas=[{"source": c["source"], "topic": c["topic"]} for c in KB],
    embeddings=EMB.tolist(),          # отдаём свои векторы той же моделью
)
print("в коллекции записей:", col.count())

In [ ]:
def chroma_search(query, k=3, where=None):
    """Поиск через Chroma. where -> фильтр по метаданным, напр. {'topic': 'возврат'}."""
    q = embedder.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k, where=where)
    out = []
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        out.append((cid, doc, meta, float(dist)))
    return out


print("ЗАПРОС: 'как оформить возврат' (с метаданными)")
for cid, doc, meta, dist in chroma_search("как оформить возврат", k=3):
    print(f"  dist={dist:.3f}  [{cid}]  source={meta['source']}")
    print(f"          {doc}")

print("\nТот же запрос, но фильтр where={'topic': 'возврат'} — поиск только по теме возврата:")
for cid, doc, meta, dist in chroma_search("как оформить возврат", k=2, where={"topic": "возврат"}):
    print(f"  [{cid}]  topic={meta['topic']}  {doc}")

`distance` у Chroma — это «дальше = хуже» (мы задали `cosine`, поэтому это `1 - cosine_similarity`). Метаданные едут вместе с чанком: по `source` дадим ссылку в ответе, через `where` отфильтруем поиск (только свежее / только доступное пользователю / только нужная тема).

## 4. Hybrid-поиск: dense + BM25

Чисто векторный поиск отлично ловит смысл и **проваливается на точных токенах**: коды ошибок, артикулы, имена, числа. На запросе «ошибка E-451» dense притащит «что-то про ошибки/оплату вообще», а не нужный чанк. Лечится это старым добрым **BM25** — лексическим поиском по словам.

Рабочий RAG гоняет оба и объединяет результаты. Простой и устойчивый способ слияния — **Reciprocal Rank Fusion (RRF)**: каждому документу начисляем `1 / (k + rank)` по каждому списку и складываем. Важно, что RRF работает с *рангами*, а не со «сырыми» очками, которые у dense и BM25 несравнимы между собой.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize(text):
    # простой токенайзер: нижний регистр + только буквы/цифры/дефис
    # (дефис важен, чтобы 'e-451' и 'sku-90210' остались одним токеном)
    import re
    return re.findall(r"[\w-]+", text.lower())


bm25 = BM25Okapi([tokenize(d) for d in DOCS])


def bm25_search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(IDS[i], DOCS[i], float(scores[i])) for i in top]


def hybrid_search(query, k=3, pool=10, rrf_k=60):
    """Сливаем dense и BM25 через Reciprocal Rank Fusion."""
    dense_ids = [cid for cid, _, _ in dense_search(query, k=pool)]
    bm25_ids = [cid for cid, _, _ in bm25_search(query, k=pool)]

    fused = {}
    for rank, cid in enumerate(dense_ids):
        fused[cid] = fused.get(cid, 0.0) + 1.0 / (rrf_k + rank)
    for rank, cid in enumerate(bm25_ids):
        fused[cid] = fused.get(cid, 0.0) + 1.0 / (rrf_k + rank)

    order = sorted(fused, key=lambda c: -fused[c])[:k]
    by_id = {c["id"]: c["text"] for c in KB}
    return [(cid, by_id[cid], fused[cid]) for cid in order]

Теперь главный демонстрационный момент: запрос с точным токеном. Сравним, что достаёт чистый dense и что — hybrid.

In [ ]:
def compare(query, k=3):
    print(f"ЗАПРОС: {query!r}\n")
    print("  DENSE (чисто векторный):")
    show(dense_search(query, k=k))
    print("\n  BM25 (лексический):")
    show(bm25_search(query, k=k))
    print("\n  HYBRID (RRF: dense + BM25):")
    for cid, text, score in hybrid_search(query, k=k):
        print(f"  rrf={score:.4f}  [{cid}]  {text}")
    print("=" * 80)


# Точный код ошибки: dense размывает, BM25 берёт дословно -> hybrid вытягивает наверх.
compare("ошибка E-451 при оплате")

# Точный артикул товара: та же история.
compare("что за товар SKU-90210")

Видно контраст: на `E-451` и `SKU-90210` чисто векторный поиск ставит нужный чанк не первым (а то и теряет из top-3) — смысл-то размазан, дословного совпадения он не чувствует. BM25 ловит точный токен мгновенно, и RRF поднимает правильный чанк наверх в hybrid. Это и есть ответ на вопрос «зачем гибрид»: dense для смысла, BM25 для точных токенов.

## 5. Reranking: cross-encoder

Поиск по эмбеддингам быстрый, но грубый: он сравнивает вектор запроса с векторами чанков **по отдельности**. **Reranker** (cross-encoder) смотрит на пару «запрос + чанк» **вместе** и оценивает релевантность точнее — но он медленный, поэтому его не пускают на всю базу.

Схема: retrieval (hybrid) достаёт top-N кандидатов дёшево → reranker переупорядочивает их и оставляет точный top-k. Часто это даёт больше прироста, чем смена БД или embedding-модели.

In [ ]:
# Cross-encoder: на вход пары (query, doc), на выход — скор релевантности.
# Модель англоязычная, но на наших коротких чанках хорошо показывает сам механизм.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, candidates, k=3):
    """candidates: list[(id, text, score)] -> переупорядоченный top-k по cross-encoder."""
    pairs = [(query, text) for _, text, _ in candidates]
    ce_scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, ce_scores), key=lambda x: -x[1])
    return [(cid, text, float(s)) for (cid, text, _), s in ranked[:k]]


query = "за сколько дней вернут деньги после возврата"

# Шаг 1: дёшево достаём широкий пул кандидатов (top-N) гибридом.
pool = hybrid_search(query, k=8, pool=10)
print(f"ЗАПРОС: {query!r}\n")
print("  ДО reranking (порядок от hybrid, top-8 кандидатов):")
show(pool)

# Шаг 2: cross-encoder переупорядочивает пул в точный top-3.
print("\n  ПОСЛЕ reranking (cross-encoder, top-3):")
show(rerank(query, pool, k=3))

Порядок до и после reranking отличается: cross-encoder поднимает чанк, который прямо отвечает на вопрос («деньги приходят в течение 3-10 рабочих дней»), даже если по гибриду он был не первым. Это финальный, самый точный отбор перед тем, как класть контекст в промпт.

## 6. Генерация ответа (нужен ключ, без ключа — аккуратно пропускается)

Финальный шаг: собираем найденные чанки в контекст, кладём в промпт и просим Claude ответить **только по нему**, сославшись на источник. Анти-галлюцинационный system-prompt: «отвечай только из контекста; если ответа нет — честно скажи, что не знаешь».

Это единственный шаг, которому нужен ключ. Читаем `ANTHROPIC_API_KEY` из окружения / Kaggle Secrets / Colab userdata. Если ключа нет — печатаем сообщение и пропускаем, чтобы `Run all` не падал.

In [ ]:
# Достаём ключ из трёх возможных мест, ничего не хардкодим.
def load_api_key():
    if os.getenv("ANTHROPIC_API_KEY"):
        return True
    # Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
        return True
    except Exception:
        pass
    # Colab userdata
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        return True
    except Exception:
        pass
    return bool(os.getenv("ANTHROPIC_API_KEY"))


HAS_KEY = load_api_key()
MODEL = "claude-haiku-4-5"   # дешёвый, для ответа по готовому контексту хватает с запасом
print("ключ найден:" , HAS_KEY, "| если False — шаг генерации ниже будет пропущен")

In [ ]:
SYSTEM_PROMPT = (
    "Ты ассистент поддержки. Отвечай ТОЛЬКО по контексту ниже. "
    "Если ответа в контексте нет — честно ответь: 'Не знаю, в базе нет ответа на этот вопрос.' "
    "Не придумывай фактов. В конце ответа укажи источник в формате [source: ...] "
    "по чанку, на который опирался."
)


def retrieve(query, k=3, pool=8):
    """Полный retrieval-конвейер: hybrid -> rerank -> top-k чанков с метаданными."""
    pool_hits = hybrid_search(query, k=pool, pool=10)
    reranked = rerank(query, pool_hits, k=k)
    meta_by_id = {c["id"]: c for c in KB}
    return [meta_by_id[cid] for cid, _, _ in reranked]


def build_context(chunks):
    # отдаём модели id + источник + текст, чтобы ей было на что сослаться
    return "\n".join(f"[{c['id']}] (source: {c['source']}) {c['text']}" for c in chunks)


def answer(query, k=3):
    chunks = retrieve(query, k=k)
    context = build_context(chunks)
    print(f"ЗАПРОС: {query!r}")
    print("найденные чанки:", [c["id"] for c in chunks])
    if not HAS_KEY:
        print("[пропуск генерации] нет ANTHROPIC_API_KEY — retrieval отработал, "
              "LLM-ответ не запрашиваем.\n")
        return None
    from anthropic import Anthropic
    client_llm = Anthropic()    # читает ANTHROPIC_API_KEY из окружения
    resp = client_llm.messages.create(
        model=MODEL,
        max_tokens=400,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {query}"}],
    )
    text = resp.content[0].text
    print("ОТВЕТ:\n" + text + "\n")
    return text


# Вопрос, ответ на который в базе ЕСТЬ -> Claude отвечает и ссылается на источник.
answer("через сколько дней вернут деньги за возврат")

# Вопрос, ответа на который в базе НЕТ -> бот должен сказать "не знаю", а не сочинять.
answer("какая сегодня фаза луны")

На вопрос из базы Claude отвечает по найденным чанкам и ставит `[source: ...]`. На вопрос вне базы («фаза луны») retrieval всё равно что-то достанет (ближайшее по смыслу), но system-prompt не даёт сочинять — модель отвечает «не знаю». Это и есть анти-галлюцинация: качество ответа держится на двух вещах — что нашёл retrieval и насколько жёстко промпт привязывает ответ к контексту.

> Если выше напечаталось «пропуск генерации» — значит ключа нет, и это нормально: retrieval-часть полностью отработала. Чтобы увидеть ответы Claude, добавьте `ANTHROPIC_API_KEY` (Kaggle: `Add-ons → Secrets`; Colab: значок ключа слева) и перезапустите.

## 7. А как понять, что стало лучше? (про evals)

Все правки выше — чанкинг, гибрид, reranker, модель — можно делать бесконечно «на глаз» и не знать, помогли ли они. RAG меряют по двум осям: качество **retrieval** (нашёлся ли вообще нужный чанк — context recall/precision) и качество **ответа** (опирается ли он на контекст и не врёт ли — faithfulness). Инструмент по умолчанию — `ragas`.

Это отдельная большая тема — [Модуль 9. Evals](https://itrubnikov.github.io/Train_of_Thought/docs/modules/09-evals/). Чтобы не тащить тяжёлую зависимость и не ломать `Run all`, оставляем здесь только заготовку «золотого набора» — закомментированный код для самопроверки retrieval без всякого ragas.

In [ ]:
# Мини-eval retrieval без внешних зависимостей: для набора вопросов проверяем,
# что нужный чанк попал в top-k. Это уже честное число вместо "вроде получше".
GOLD = [
    {"q": "ошибка E-451 при оплате", "expected": "pay-03"},
    {"q": "что за наушники SKU-90210", "expected": "prod-01"},
    {"q": "за сколько вернут деньги", "expected": "ret-02"},
    {"q": "сколько стоит доставка", "expected": "del-01"},
]

hits = 0
for case in GOLD:
    got = [c["id"] for c in retrieve(case["q"], k=3)]
    ok = case["expected"] in got
    hits += ok
    print(f"{'OK ' if ok else 'MISS'}  q={case['q']!r:42}  ждали {case['expected']}, top-3={got}")
print(f"\nrecall@3 = {hits}/{len(GOLD)} = {hits / len(GOLD):.2f}")

# Полноценные метрики (faithfulness, context precision) с LLM-судьёй -> ragas, модуль 9:
# !pip install -q ragas
# from ragas import evaluate
# ... (нужен ключ и заметно больше времени — здесь не запускаем)

## Задачи — доработайте рабочий пайплайн

Пайплайн целиком работает. Теперь учимся, меняя его под свои данные. Каждая задача проверяется вами самостоятельно.

1. **Своя база.** Замените `KB` на свои 15-30 коротких чанков из реальных текстов: заметки, README, FAQ продукта, конспект. Не игрушечные про доставку. Заполните `id`, `text`, `source`, `topic`. После замены перезапустите ячейки с `EMB`, `col.add(...)` и `bm25`, чтобы и dense, и Chroma, и BM25 индексировали новую базу.

2. **Промах dense → ловит hybrid.** Найдите в своей базе вопрос с точным токеном (название/код/артикул/имя), на котором `dense_search` ставит нужный чанк не первым (или теряет), а `hybrid_search` вытягивает наверх. Прогоните `compare(ваш_запрос)` и в markdown-ячейке запишите: запрос, что вернул dense, что вернул hybrid.

3. **Ответ со ссылкой + метаданные.** Убедитесь, что в ваших чанках заполнен `source`, и что `answer(...)` в ответе печатает `[source: ...]` на правильный чанк. Если ключа нет — достаточно показать, что `retrieve(...)` достаёт правильный чанк и `build_context` кладёт его источник в контекст.

4. **Анти-галлюцинация.** Задайте `answer(...)` вопрос, ответа на который в вашей базе точно нет, и убедитесь, что бот отвечает «не знаю», а не сочиняет. Запишите вопрос и ответ.

Минимум для сдачи — задачи 1, 2, 4. Задача 3 — если есть ключ (иначе показываете retrieval-часть).

In [ ]:
# Задача 1: вставьте сюда свою базу (раскомментируйте и заполните),
# затем перезапустите ячейки с EMB, col.add(...) и bm25 выше.
#
# KB = [
#     {"id": "my-01", "text": "...", "source": "notes/...", "topic": "..."},
#     ...  # 15-30 ваших чанков
# ]
# DOCS = [c["text"] for c in KB]
# IDS = [c["id"] for c in KB]
print("Задача 1: замените KB на свою базу и переиндексируйте (EMB, Chroma, BM25).")

In [ ]:
# Задача 2: ваш запрос, где dense мажет, а hybrid ловит.
# compare("ваш запрос с точным токеном")
print("Задача 2: вызовите compare(...) на своём запросе и сравните dense vs hybrid.")

**Задача 2 — наблюдение (заполните):**

- запрос:
- что вернул dense (top-3 id):
- что вернул hybrid (top-3 id):
- вывод (почему dense промахнулся):

In [ ]:
# Задача 3 + 4: ответ со ссылкой и честное "не знаю".
# answer("вопрос, ответ на который ЕСТЬ в вашей базе")
# answer("вопрос, ответа на который в базе НЕТ")
print("Задачи 3-4: проверьте ссылку на источник и поведение 'не знаю'.")

**Задача 4 — анти-галлюцинация (заполните):**

- вопрос вне базы:
- что ответил бот:
- сослался ли на источник на вопрос из базы (id):

## Что сдать

Самопроверка — всё проверяется без преподавателя:

- [ ] Ноутбук прогнан целиком (`Run all`) — retrieval-часть отработала без ключа, вывод виден.
- [ ] `KB` заменена на вашу базу из 15-30 реальных чанков (не демо про доставку).
- [ ] Найден и записан запрос, где dense-поиск мажет, а hybrid вытягивает нужный чанк.
- [ ] `answer(...)` ссылается на источник (`[source: ...]`); в чанках есть метаданные (если есть ключ).
- [ ] На вопрос вне базы бот отвечает «не знаю», а не выдумывает.
- [ ] Ключ нигде не захардкожен — только `.env` / Secrets / userdata.

**Артефакт** — публичная ссылка на прогнанный Kaggle/Colab-ноутбук, где видно вашу базу, удачный retrieval, один промах dense vs hybrid и честное «не знаю». Пришлите в чат курса как `[Модуль 7, ДЗ 1] {ссылка}`.